In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, auc, f1_score, confusion_matrix

# 1. Load Data 
csv_path = r"C:\Users\user\fraud-detection\data\processed\cleaned_fraud_data.csv"
print(f"📂 Loading data for Task 2 Modeling from: {csv_path}")
df = pd.read_csv(csv_path)

# Separate features/target and filter for numeric columns
X = df.drop(columns=['class']) if 'class' in df.columns else df.drop(columns=[df.columns[-1]])
y = df['class'] if 'class' in df.columns else df[df.columns[-1]]
X = X.select_dtypes(include=[np.number])

# Initial Stratified Split to get a clean holdout test set
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (Critical for Logistic Regression convergence!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data prepped. Full Training Shape: {X_train_full.shape}")

# 2. Define our 5-Fold Cross-Validation Protocol
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_with_cv(model_type, X_data, y_data, is_scaled=False):
    """Executes a 5-Fold Stratified CV loop and reports AUC-PR and F1 scores."""
    f1_scores = []
    auc_pr_scores = []
    
    print(f"\n🔄 Running 5-Fold Stratified CV for {model_type}...")
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data), 1):
        # Handle indexing for numpy scaled arrays vs pandas dataframes
        if is_scaled:
            X_tr, X_val = X_data[train_idx], X_data[val_idx]
        else:
            X_tr, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
            
        y_tr, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]
        
        # Initialize model instances based on requirements
        if model_type == "Logistic Regression (Baseline)":
            model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
        elif model_type == "Random Forest (Ensemble)":
            model = RandomForestClassifier(n_estimators=50, max_depth=8, class_weight='balanced', n_jobs=-1, random_state=42)
            
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        probs = model.predict_proba(X_val)[:, 1]
        
        # Calculate AUC-PR (Area Under Precision-Recall Curve)
        precision, recall, _ = precision_recall_curve(y_val, probs)
        auc_pr = auc(recall, precision)
        
        f1_scores.append(f1_score(y_val, preds))
        auc_pr_scores.append(auc_pr)
        
    print(f"   ↳ F1-Score: {np.mean(f1_scores):.4f} (±{np.std(f1_scores):.4f})")
    print(f"   ↳ AUC-PR:   {np.mean(auc_pr_scores):.4f} (±{np.std(auc_pr_scores):.4f})")
    return np.mean(f1_scores), np.mean(auc_pr_scores)

# Run Cross Validation across both architectures
lr_f1, lr_auc = evaluate_with_cv("Logistic Regression (Baseline)", X_train_scaled, y_train_full, is_scaled=True)
rf_f1, rf_auc = evaluate_with_cv("Random Forest (Ensemble)", X_train_full, y_train_full, is_scaled=False)

📂 Loading data for Task 2 Modeling from: C:\Users\user\fraud-detection\data\processed\cleaned_fraud_data.csv
✅ Data prepped. Full Training Shape: (120889, 9)

🔄 Running 5-Fold Stratified CV for Logistic Regression (Baseline)...
   ↳ F1-Score: 0.6075 (±0.0022)
   ↳ AUC-PR:   0.6617 (±0.0075)

🔄 Running 5-Fold Stratified CV for Random Forest (Ensemble)...
   ↳ F1-Score: 0.6208 (±0.0037)
   ↳ AUC-PR:   0.7191 (±0.0054)
